# Chapter 6: Memory Systems


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 6.1 Context Optimization


In [ ]:
from scratch_agent.memory.context_optimizer import (
    apply_sliding_window, apply_compaction, count_tokens
)
from scratch_agent.llm import LlmRequest
from scratch_agent.types import Message, ToolCall, ToolResult
from scratch_agent.context import ExecutionContext

# Build a sample LlmRequest with many messages
contents = [Message(role="user", content="What is AI?")] + [
    Message(role="assistant", content=f"Response about topic {i}. " * 50)
    for i in range(20)
]

request = LlmRequest(
    instructions=["You are a helpful assistant."],
    contents=contents,
)
print(f"Original content items: {len(request.contents)}")
print(f"Approximate tokens: {count_tokens(request)}")

# Apply sliding window
context = ExecutionContext()
apply_sliding_window(context, request, window_size=5)
print(f"\nAfter sliding window: {len(request.contents)} items")
print(f"Approximate tokens: {count_tokens(request)}")

## 6.2 Token Counting


In [ ]:
from scratch_agent.memory.context_optimizer import count_tokens
from scratch_agent.llm import LlmRequest
from scratch_agent.types import Message

# Demo token counting with different text lengths
test_messages = [
    "Hello, world!",
    "This is a longer sentence that should have more tokens.",
    "Artificial intelligence and machine learning are transforming industries worldwide.",
]

for text in test_messages:
    request = LlmRequest(
        contents=[Message(role="user", content=text)],
    )
    tokens = count_tokens(request)
    print(f"{tokens:4d} tokens: {text}")

## 6.3 ContextOptimizer


In [ ]:
from scratch_agent.memory.context_optimizer import ContextOptimizer
from scratch_agent.llm import LlmClient

# Create a ContextOptimizer (used as a before_llm_callback)
llm = LlmClient(model="gpt-4o-mini")
optimizer = ContextOptimizer(
    llm_client=llm,
    token_threshold=50000,
    enable_compaction=True,
    enable_summarization=True,
    keep_recent=5,
)

print(f"Token threshold: {optimizer.token_threshold}")
print(f"Compaction enabled: {optimizer.enable_compaction}")
print(f"Summarization enabled: {optimizer.enable_summarization}")
print("The optimizer is registered as a before_llm_callback on the Agent.")

## 6.4 Session Management


In [ ]:
from scratch_agent.memory.session import Session, InMemorySessionManager

# Create a session manager
session_manager = InMemorySessionManager()

# Create a new session
session = await session_manager.create("demo-session-001")
print(f"Session ID: {session.session_id}")
print(f"Events: {len(session.events)}")
print(f"State: {session.state}")

# Retrieve the session
retrieved = await session_manager.get("demo-session-001")
print(f"\nRetrieved session: {retrieved.session_id}")

# get_or_create - returns existing
same = await session_manager.get_or_create("demo-session-001")
print(f"Same session: {same.session_id == session.session_id}")

## 6.5 Multi-turn Conversation


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.memory.session import InMemorySessionManager
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Create an agent with session support
session_mgr = InMemorySessionManager()
agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator],
    instructions="You are a helpful assistant with memory. Remember what users tell you.",
    session_manager=session_mgr,
)

# Multi-turn conversation using session_id
session_id = "demo-session-001"

result1 = await agent.run("My name is Alice.", session_id=session_id)
print(f"Turn 1: {result1.output}")

result2 = await agent.run("What is my name?", session_id=session_id)
print(f"Turn 2: {result2.output}")

## 6.6 Human-in-the-Loop


In [ ]:
from scratch_agent.context import PendingToolCall, ToolConfirmation
from scratch_agent.types import ToolCall
from scratch_agent.tools.base import FunctionTool

# Create a tool that requires confirmation
def delete_file(filename: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {filename}"

delete_tool = FunctionTool(
    func=delete_file,
    requires_confirmation=True,
    confirmation_message_template="Are you sure you want to delete {arguments}?",
)

print(f"Tool: {delete_tool.name}")
print(f"Requires confirmation: {delete_tool.requires_confirmation}")

# Demo PendingToolCall
tc = ToolCall(tool_call_id="call_001", name="delete_file", arguments={"filename": "test.txt"})
pending = PendingToolCall(
    tool_call=tc,
    confirmation_message=delete_tool.get_confirmation_message(tc.arguments),
)
print(f"\nPending call: {pending}")

# Demo ToolConfirmation
confirmation = ToolConfirmation(tool_call_id="call_001", approved=True)
print(f"Confirmation: {confirmation}")

## 6.7 Long-term Memory


In [ ]:
from scratch_agent.memory.long_term import TaskMemory, TaskMemoryManager
from scratch_agent.tools.memory_tool import MemoryTool
from scratch_agent.llm import LlmClient

# Create a task memory manager
llm = LlmClient(model="gpt-4o-mini")
memory_manager = TaskMemoryManager(llm_client=llm)

# Inspect the memory tool
memory_tool = MemoryTool(memory_manager)
print(f"Memory tool: {memory_tool.name}")
print(f"Description: {memory_tool.description}")

# Demo TaskMemory structure
sample_memory = TaskMemory(
    task_summary="Calculate the distance between Earth and Moon",
    approach="Used calculator tool to compute distance at closest approach",
    final_answer="363,300 km",
    is_correct=True,
)
print(f"\nSample memory:")
print(f"  Task: {sample_memory.task_summary}")
print(f"  Approach: {sample_memory.approach}")
print(f"  Answer: {sample_memory.final_answer}")
print(f"\nEmbedding text:\n{sample_memory.to_embedding_text()}")